# Training vs Generated Data Analysis

Compares aggregate statistics, distributions and temporal structure between:
- **Reference**: the first `TRAINING_DATASET_SIZE` CSVs from the training directory (sorted)
- **Generated**: all CSVs produced by the reverse-diffusion model

Both datasets are either log-prices or log-returns.
Reference paths may be full-length stock histories; generated paths are fixed-length windows.

In [ ]:
# ── USER INPUTS ───────────────────────────────────────────────────────────────
REF_DIRECTORY         = "../data/replication_returns_other"          # training data folder
GEN_DIRECTORY         = "../data/generated/replication/ODE_NO_NO_REPL_RET_OTHER_UNCO_ep-400_sde-ve_noise-exponential_20260516_205659_N2000_seed50_seqlen_512_stride100"
SEED                  = 50
TRAINING_SEED         = 50    # must match config["train"]["seed"] used during training
                               
TRAINING_DATASET_SIZE = None
VAL_SPLIT_RATIO = 0.05# set to None to use all available (post-val) windows
SEQ_LEN               = 512  # window length used during training (--seq_len)
STRIDE                = 100   # stride used during training (--stride)
# ─────────────────────────────────────────────────────────────────────────────

In [94]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf as sm_acf

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

print(f"REF_DIRECTORY         : {REF_DIRECTORY}")
print(f"GEN_DIRECTORY         : {GEN_DIRECTORY}")
print(f"SEED                  : {SEED}")
print(f"TRAINING_DATASET_SIZE : {TRAINING_DATASET_SIZE}")
print(f"SEQ_LEN               : {SEQ_LEN}")
print(f"STRIDE                : {STRIDE}")

REF_DIRECTORY         : ../data/replication_returns_other
GEN_DIRECTORY         : ../data/generated/replication/ODE_NO_NO_REPL_RET_OTHER_UNCO_ep-400_sde-ve_noise-exponential_20260516_205659_N2000_seed50_seqlen_512_stride100
SEED                  : 50
TRAINING_DATASET_SIZE : None
SEQ_LEN               : 512
STRIDE                : 100


## 1. Load data

In [95]:
# ── Reference data ────────────────────────────────────────────────────────────
# Mirrors csdi_train_modified.py exactly:
#   1. Load ALL sorted CSVs — no file cap
#   2. Extract sliding windows with same SEQ_LEN / STRIDE / drop_incomplete=True
#   3. Replicate torch.randperm(seed) + val split + train_subset_size
import torch

ref_csv_files = sorted(Path(REF_DIRECTORY).glob("*.csv"))
if len(ref_csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {REF_DIRECTORY}")

all_windows: list[np.ndarray] = []
n_skipped_short = 0
for fp in ref_csv_files:
    series = pd.read_csv(fp)["log_adj_close"].values.astype(np.float64)
    T = len(series)
    max_start = T - SEQ_LEN
    if max_start < 0:   # shorter than SEQ_LEN — skip (drop_incomplete=True)
        n_skipped_short += 1
        continue
    for s in range(0, max_start + 1, STRIDE):
        all_windows.append(series[s : s + SEQ_LEN])

if n_skipped_short:
    print(f"[WARNING] {n_skipped_short} file(s) shorter than SEQ_LEN={SEQ_LEN} were skipped.")
if len(all_windows) == 0:
    raise RuntimeError("No windows could be extracted. Check SEQ_LEN / STRIDE / REF_DIRECTORY.")

print(f"Total windows in full dataset : {len(all_windows)} "
      f"(from {len(ref_csv_files) - n_skipped_short} usable file(s), "
      f"SEQ_LEN={SEQ_LEN}, STRIDE={STRIDE})")

# ── Replicate training subset selection with torch RNG ────────────────────────
torch_rng  = torch.Generator().manual_seed(TRAINING_SEED)
all_indices = torch.randperm(len(all_windows), generator=torch_rng).tolist()

# Carve out val split first (mirrors val_split_ratio logic in training script)
if VAL_SPLIT_RATIO is not None:
    val_size   = max(1, int(len(all_windows) * VAL_SPLIT_RATIO))
    train_pool = all_indices[val_size:]
    print(f"Val split : {val_size} windows held out (VAL_SPLIT_RATIO={VAL_SPLIT_RATIO})")
else:
    train_pool = all_indices

# Then take TRAINING_DATASET_SIZE from the remaining pool
if TRAINING_DATASET_SIZE is not None and TRAINING_DATASET_SIZE < len(train_pool):
    train_indices = train_pool[:TRAINING_DATASET_SIZE]
    print(f"Reference : {len(train_indices)} windows "
          f"(TRAINING_DATASET_SIZE={TRAINING_DATASET_SIZE} of {len(train_pool)} in pool)")
else:
    train_indices = train_pool
    if TRAINING_DATASET_SIZE is not None:
        print(f"[WARNING] TRAINING_DATASET_SIZE={TRAINING_DATASET_SIZE} >= pool size "
              f"({len(train_pool)}); using all pool windows.")
    print(f"Reference : {len(train_indices)} windows (full training pool)")

ref_paths = [all_windows[i] for i in train_indices]

assert all(len(w) == SEQ_LEN for w in ref_paths), "Window length mismatch — check SEQ_LEN"
print(f"  All window lengths = {SEQ_LEN}  ✓")

Total windows in full dataset : 25225 (from 210 usable file(s), SEQ_LEN=512, STRIDE=100)
Val split : 1261 windows held out (VAL_SPLIT_RATIO=0.05)
Reference : 23964 windows (full training pool)
  All window lengths = 512  ✓


In [96]:
# ── Generated data ────────────────────────────────────────────────────────────
# Load ALL CSV files in GEN_DIRECTORY (no size cap).
# Two formats are supported:
#   (a) Per-sample CSVs  — columns: date, log_adj_close  (one file per path)
#   (b) Wide-format CSV  — columns: sample_idx, start_date, end_date, step_000, step_001, ...
#       (produced by generate_samples.py as "generated_samples.csv")
gen_csv_files = sorted(Path(GEN_DIRECTORY).glob("*.csv"))

if len(gen_csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {GEN_DIRECTORY}")

gen_paths: list[np.ndarray] = []
skipped = 0
for fp in gen_csv_files:
    df = pd.read_csv(fp)
    if "log_adj_close" in df.columns:
        # Format (a): individual per-sample CSV
        gen_paths.append(df["log_adj_close"].values.astype(np.float64))
    else:
        # Format (b): wide-format — each row is one sample path
        step_cols = sorted(
            [c for c in df.columns if c.startswith("step_")],
            key=lambda c: int(c.split("_")[1])
        )
        print(step_cols[999:], f"\n first column: {step_cols[0]}, last column: {step_cols[-1]}")
        if step_cols:
            print(df.head())
            for _, row in df.iterrows():
                gen_paths.append(row[step_cols].values.astype(np.float64))
        else:
            skipped += 1

if skipped:
    print(f"[WARNING] Skipped {skipped} file(s) with unrecognised column layout.")
if len(gen_paths) == 0:
    raise RuntimeError("No valid generated paths could be loaded from GEN_DIRECTORY.")

gen_lens = [len(p) for p in gen_paths]
print(f"Generated: {len(gen_paths)} paths loaded  ({len(gen_csv_files)} CSV file(s) read)")
print(f"  Path lengths — min={min(gen_lens)}  max={max(gen_lens)}  "
      f"median={int(np.median(gen_lens))}")

print(gen_paths[0][:5], "\n", gen_paths[0][-5:])
print(gen_paths[-1][:5], "\n", gen_paths[-1][-5:])

[] 
 first column: step_000, last column: step_511
   sample_idx  start_date    end_date  step_000  step_001  step_002  step_003  \
0           0  2013-12-18  2015-12-30  0.004980 -0.007942  0.000401  0.002676   
1           1  2016-04-18  2018-04-27  0.006835  0.022342 -0.014950 -0.006817   
2           2  2002-07-12  2004-07-23 -0.008511 -0.002055 -0.007886  0.042288   
3           3  2022-11-08  2024-11-20  0.018548 -0.020888  0.017285 -0.054469   
4           4  1987-10-13  1989-10-19  0.004092  0.001922  0.035601 -0.028314   

   step_004  step_005  step_006  ...  step_502  step_503  step_504  step_505  \
0  0.005093  0.002448  0.008932  ...  0.004182 -0.015419 -0.002204  0.014117   
1  0.031503 -0.002941  0.013313  ...  0.042304  0.016561 -0.017541 -0.018761   
2  0.015988 -0.054214 -0.057521  ... -0.000173 -0.008791  0.015711 -0.026691   
3 -0.008023  0.006344 -0.004707  ...  0.016900 -0.003185  0.006836 -0.014855   
4 -0.063878 -0.053917  0.064464  ... -0.010523  0.000558 -0.01

## 2. Aggregate statistics

In [97]:
def compute_stats(paths: list[np.ndarray], label: str) -> dict:
    flat = np.concatenate(paths)
    incs = np.concatenate([np.diff(p) for p in paths if len(p) > 1])
    return {
        "label"            : label,
        "n_paths"          : len(paths),
        "n_values"         : len(flat),
        "mean"             : flat.mean(),
        "std"              : flat.std(),
        "min"              : flat.min(),
        "q01"              : np.quantile(flat, 0.01),
        "q05"              : np.quantile(flat, 0.05),
        "q25"              : np.quantile(flat, 0.25),
        "q50"              : np.quantile(flat, 0.50),
        "q75"              : np.quantile(flat, 0.75),
        "q95"              : np.quantile(flat, 0.95),
        "q99"              : np.quantile(flat, 0.99),
        "max"              : flat.max(),
        "skewness"         : scipy_stats.skew(flat),
        "excess_kurtosis"  : scipy_stats.kurtosis(flat, fisher=True),
        "inc_mean"         : incs.mean(),
        "inc_std"          : incs.std(),
        "inc_min"          : incs.min(),
        "inc_q01"          : np.quantile(incs, 0.01),
        "inc_q99"          : np.quantile(incs, 0.99),
        "inc_max"          : incs.max(),
        "inc_skewness"     : scipy_stats.skew(incs),
        "inc_excess_kurt"  : scipy_stats.kurtosis(incs, fisher=True),
    }

ref_stats = compute_stats(ref_paths, "Reference (training)")
gen_stats = compute_stats(gen_paths, "Generated")

# ── Levels table ──────────────────────────────────────────────────────────────
level_keys = ["n_paths", "n_values", "mean", "std", "min", "q01", "q05",
              "q25", "q50", "q75", "q95", "q99", "max", "skewness", "excess_kurtosis"]
df_levels = pd.DataFrame(
    {k: {"Reference": ref_stats[k], "Generated": gen_stats[k]} for k in level_keys}
).T.round(4)

print("=" * 60)
print("  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)")
print("=" * 60)
display(df_levels.style.format("{:.4f}"))

  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)


,Reference,Generated
n_paths,23964.0000,512.0000
n_values,12269568.0000,262144.0000
mean,0.0005,0.0005
std,0.0209,0.0201
min,-0.9363,-0.3303
q01,-0.0562,-0.0510
q05,-0.0292,-0.0303
q25,-0.0086,-0.0110
q50,0.0000,0.0004
q75,0.0095,0.0119


In [98]:
for paths, label in [(ref_paths, "Reference"), (gen_paths, "GenWerated")]:
    flat = np.concatenate(paths)
    zero_pct = 100.0 * np.mean(flat == 0.0)
    print(f"{label}: {zero_pct:.4f}% absolute zeros ({(flat == 0.0).sum()} / {len(flat)})")


Reference: 7.5499% absolute zeros (926346 / 12269568)
GenWerated: 0.0000% absolute zeros (0 / 262144)


In [99]:
# ── Increments table ──────────────────────────────────────────────────────────
inc_keys = ["inc_mean", "inc_std", "inc_min", "inc_q01", "inc_q99",
            "inc_max", "inc_skewness", "inc_excess_kurt"]
rename = {
    "inc_mean": "mean", "inc_std": "std", "inc_min": "min",
    "inc_q01": "q01", "inc_q99": "q99", "inc_max": "max",
    "inc_skewness": "skewness", "inc_excess_kurt": "excess_kurtosis",
}
df_incs = pd.DataFrame(
    {rename[k]: {"Reference": ref_stats[k], "Generated": gen_stats[k]}
     for k in inc_keys}
).T.round(6)

print("=" * 60)
print("  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics")
print("=" * 60)
display(df_incs.style.format("{:.4f}"))

  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics


,Reference,Generated
mean,0.0000,0.0000
std,0.0297,0.0284
min,-1.3863,-0.3852
q01,-0.0804,-0.0719
q99,0.0818,0.0727
max,1.0291,0.5654
skewness,0.2328,0.1772
excess_kurtosis,30.5751,6.2086


## 2b. Aggregate statistics — full stock series vs generated

In [100]:
# Load full stock series from REF_DIRECTORY (one per stock, no windowing)
ref_full_paths = []
for fp in sorted(Path(REF_DIRECTORY).glob("*.csv")):
    series = pd.read_csv(fp)["log_adj_close"].values.astype(np.float64)
    ref_full_paths.append(series)

print(f"Full stock series loaded : {len(ref_full_paths)}")
print(f"Lengths — min={min(len(s) for s in ref_full_paths)}  "
      f"max={max(len(s) for s in ref_full_paths)}  "
      f"mean={np.mean([len(s) for s in ref_full_paths]):.0f}")

ref_full_stats = compute_stats(ref_full_paths, "Reference (full series)")

# ── Levels table ──────────────────────────────────────────────────────────────
level_keys = ["n_paths", "n_values", "mean", "std", "min", "q01", "q05",
              "q25", "q50", "q75", "q95", "q99", "max", "skewness", "excess_kurtosis"]
df_levels_full = pd.DataFrame(
    {k: {"Reference (full)": ref_full_stats[k], "Generated": gen_stats[k]} for k in level_keys}
).T.round(4)

print("\n" + "=" * 60)
print("  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)")
print("=" * 60)
display(df_levels_full.style.format("{:.4f}"))

# ── Increments table ──────────────────────────────────────────────────────────
inc_keys = ["inc_mean", "inc_std", "inc_min", "inc_q01", "inc_q99",
            "inc_max", "inc_skewness", "inc_excess_kurt"]
rename = {
    "inc_mean": "mean", "inc_std": "std", "inc_min": "min",
    "inc_q01": "q01", "inc_q99": "q99", "inc_max": "max",
    "inc_skewness": "skewness", "inc_excess_kurt": "excess_kurtosis",
}
df_incs_full = pd.DataFrame(
    {rename[k]: {"Reference (full)": ref_full_stats[k], "Generated": gen_stats[k]}
     for k in inc_keys}
).T.round(6)

print("\n" + "=" * 60)
print("  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics")
print("=" * 60)
display(df_incs_full.style.format("{:.4f}"))

Full stock series loaded : 210
Lengths — min=10083  max=16175  mean=12493

  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)


,Reference (full),Generated
n_paths,210.0000,512.0000
n_values,2623490.0000,262144.0000
mean,0.0005,0.0005
std,0.0209,0.0201
min,-0.9363,-0.3303
q01,-0.0564,-0.0510
q05,-0.0292,-0.0303
q25,-0.0086,-0.0110
q50,0.0000,0.0004
q75,0.0095,0.0119



  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics


,Reference (full),Generated
mean,-0.0000,0.0000
std,0.0297,0.0284
min,-1.3863,-0.3852
q01,-0.0805,-0.0719
q99,0.0820,0.0727
max,1.0291,0.5654
skewness,0.2227,0.1772
excess_kurtosis,29.4105,6.2086


## Per-window statistics

In [101]:
def per_window_stats(paths: list[np.ndarray]) -> pd.DataFrame:
    rows = []
    for p in paths:
        incs = np.diff(p)
        rows.append({
            "mean"         : float(p.mean()),
            "std"          : float(p.std()),
            "skewness"     : float(scipy_stats.skew(p)),
            "excess_kurt"  : float(scipy_stats.kurtosis(p, fisher=True)),
            "zero_ratio"   : float(np.mean(p == 0.0)),
            "inc_std"      : float(incs.std()),
            "inc_skewness" : float(scipy_stats.skew(incs)),
            "inc_excess_kurt": float(scipy_stats.kurtosis(incs, fisher=True)),
        })
    return pd.DataFrame(rows)

ref_pw = per_window_stats(ref_paths)
gen_pw = per_window_stats(gen_paths)

# ── Summary table: mean ± std of each per-window stat ────────────────────────
summary = pd.DataFrame({
    "Reference  mean": ref_pw.mean(),
    "Reference  std" : ref_pw.std(),
    "Generated  mean": gen_pw.mean(),
    "Generated  std" : gen_pw.std(),
}).round(5)
print("Per-window statistics — summary (mean ± std across windows)")
display(summary.style.format("{:.4f}"))

# ── Violin plots ──────────────────────────────────────────────────────────────
metrics = list(ref_pw.columns)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for ax, metric in zip(axes.flat, metrics):
    data  = [ref_pw[metric].values, gen_pw[metric].values]
    parts = ax.violinplot(data, positions=[0, 1], showmedians=True, showextrema=True)
    parts["bodies"][0].set_facecolor("tab:blue");   parts["bodies"][0].set_alpha(0.55)
    parts["bodies"][1].set_facecolor("tab:orange"); parts["bodies"][1].set_alpha(0.55)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Reference", "Generated"])
    ax.set_title(metric)
    ax.grid(axis="y", linewidth=0.4)

plt.suptitle("Distribution of per-window statistics", y=1.02)
plt.tight_layout()
plt.show()


Per-window statistics — summary (mean ± std across windows)


,Reference mean,Reference std,Generated mean,Generated std
mean,0.0004,0.0008,0.0004,0.0008
std,0.0190,0.0086,0.0194,0.0048
skewness,-0.1749,1.0919,-0.0621,0.5365
excess_kurt,6.6514,15.6724,2.7199,5.4279
zero_ratio,0.0755,0.1096,0.0000,0.0000
inc_std,0.0269,0.0125,0.0275,0.0070
inc_skewness,0.1252,0.4465,0.0829,0.2893
inc_excess_kurt,4.8484,10.1728,2.1600,4.4152


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\2342577457.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Kurtosis offenders

The top offenders are identified through computation of the increments so as to identify most leptokurtic distribution, which have very fat tails and sharp peaks.

In [102]:
# ── Kurtosis diagnosis: worst windows by increment excess kurtosis ─────────────
TOP_K      = 10   # number of worst windows to inspect
TOP_N_INCS = 5    # largest increments to report per window

def print_worst_kurtosis_windows(pw: pd.DataFrame, paths: list, label: str,
                                  top_k: int, top_n: int) -> None:
    worst = pw.nlargest(top_k, "inc_excess_kurt")
    print(f"{'='*70}")
    print(f"Top-{top_k} {label} windows by increment excess kurtosis")
    print(f"{'='*70}\n")
    for rank, (win_idx, row) in enumerate(worst.iterrows(), start=1):
        path     = paths[win_idx]
        incs     = np.diff(path)
        abs_incs = np.abs(incs)
        print(f"Rank {rank:2d} | window index {win_idx}")
        print(f"  {'mean':<20s}: {row['mean']:+.6f}")
        print(f"  {'std':<20s}: {row['std']:.6f}")
        print(f"  {'skewness':<20s}: {row['skewness']:+.4f}")
        print(f"  {'excess_kurt (levels)':<20s}: {row['excess_kurt']:+.4f}")
        print(f"  {'inc_std':<20s}: {row['inc_std']:.6f}")
        print(f"  {'inc_skewness':<20s}: {row['inc_skewness']:+.4f}")
        print(f"  {'inc_excess_kurt':<20s}: {row['inc_excess_kurt']:+.4f}")
        top_pos = np.argsort(abs_incs)[::-1][:top_n]
        print(f"\n  Top-{top_n} largest |Δx_t|  (position l in [0, {len(incs)-1}]):")
        print(f"  {'l':>6}  {'Δx':>12}  {'|Δx|':>10}  {'x[l]':>12}  {'x[l+1]':>12}")
        for pos in top_pos:
            print(f"  {pos:>6d}  {incs[pos]:>+12.6f}  {abs_incs[pos]:>10.6f}"
                  f"  {path[pos]:>+12.6f}  {path[pos+1]:>+12.6f}")
        print()

print_worst_kurtosis_windows(gen_pw, gen_paths, "Generated", TOP_K, TOP_N_INCS)
print_worst_kurtosis_windows(ref_pw, ref_paths, "Reference", TOP_K, TOP_N_INCS)

Top-10 Generated windows by increment excess kurtosis

Rank  1 | window index 363
  mean                : -0.000728
  std                 : 0.023410
  skewness            : -5.6248
  excess_kurt (levels): +75.0761
  inc_std             : 0.031302
  inc_skewness        : +0.2954
  inc_excess_kurt     : +40.1567

  Top-5 largest |Δx_t|  (position l in [0, 510]):
       l            Δx        |Δx|          x[l]        x[l+1]
     250     +0.322932    0.322932     -0.329191     -0.006259
     249     -0.311704    0.311704     -0.017487     -0.329191
     236     +0.122496    0.122496     -0.080129     +0.042368
     352     -0.110583    0.110583     +0.002601     -0.107982
     353     +0.082043    0.082043     -0.107982     -0.025939

Rank  2 | window index 24
  mean                : +0.001147
  std                 : 0.019373
  skewness            : +0.3752
  excess_kurt (levels): +16.3246
  inc_std             : 0.027961
  inc_skewness        : +2.0521
  inc_excess_kurt     : +32.5616

 

In [103]:
# if TRAINING_DATASET_SIZE > 15 or TRAINING_DATASET_SIZE == 1.0:
# ── Aggregate statistics after removing the N_DROP most leptokurtic windows ───
N_DROP = 15   # number of worst-kurtosis windows to exclude from each dataset

gen_drop_idx = set(gen_pw.nlargest(N_DROP, "inc_excess_kurt").index)
ref_drop_idx = set(ref_pw.nlargest(N_DROP, "inc_excess_kurt").index)

gen_paths_trim = [p for i, p in enumerate(gen_paths) if i not in gen_drop_idx]
ref_paths_trim = [p for i, p in enumerate(ref_paths) if i not in ref_drop_idx]

print(f"Generated : {len(gen_paths)} → {len(gen_paths_trim)} windows "
      f"(removed {len(gen_drop_idx)} most leptokurtic)")
print(f"Reference : {len(ref_paths)} → {len(ref_paths_trim)} windows "
      f"(removed {len(ref_drop_idx)} most leptokurtic)\n")

ref_stats_trim = compute_stats(ref_paths_trim, "Reference (trimmed)")
gen_stats_trim = compute_stats(gen_paths_trim, "Generated (trimmed)")

keys = ["mean", "std", "min", "q01", "q99", "max", "skewness", "excess_kurtosis",
      "inc_mean", "inc_std", "inc_min", "inc_q01", "inc_q99", "inc_max",
      "inc_skewness", "inc_excess_kurt"]

df_cmp = pd.DataFrame({
"Gen  full"   : {k: gen_stats[k]      for k in keys},
"Gen  trimmed": {k: gen_stats_trim[k] for k in keys},
"Ref  full"   : {k: ref_stats[k]      for k in keys},
"Ref  trimmed": {k: ref_stats_trim[k] for k in keys},
}).round(6)

print(f"Aggregate statistics — full vs trimmed (top-{N_DROP} leptokurtic windows removed)")
display(df_cmp.style.format("{:.6f}"))

Generated : 512 → 497 windows (removed 15 most leptokurtic)
Reference : 23964 → 23949 windows (removed 15 most leptokurtic)

Aggregate statistics — full vs trimmed (top-15 leptokurtic windows removed)


,Gen full,Gen trimmed,Ref full,Ref trimmed
mean,0.000453,0.000470,0.000454,0.000455
std,0.020060,0.019867,0.020863,0.020854
min,-0.330341,-0.274799,-0.936258,-0.936258
q01,-0.050982,-0.050584,-0.056240,-0.056240
q99,0.052239,0.051930,0.058326,0.058335
max,0.260784,0.260784,0.693147,0.693147
skewness,-0.124103,-0.033125,-0.591644,-0.562543
excess_kurtosis,6.489608,4.804591,40.132844,38.941565
inc_mean,0.000002,0.000002,0.000002,0.000002
inc_std,0.028424,0.028135,0.029681,0.029665


In [104]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (pw, paths, label, color) in zip(axes, [
    (ref_pw, ref_paths, "Reference", "tab:blue"),
    (gen_pw, gen_paths, "Generated", "tab:orange"),
]):
    worst_idx = pw["inc_excess_kurt"].idxmax()
    p = paths[worst_idx]
    ax.plot(p, linewidth=0.8, color=color)
    ax.set_title(f"{label} — most leptokurtic window\n"
                 f"inc_excess_kurt={pw.loc[worst_idx, 'inc_excess_kurt']:.2f}  "
                 f"idx={worst_idx}")
    ax.set_xlabel("step t")
    ax.set_ylabel("log_adj_close")
    ax.grid(True, linewidth=0.3)

plt.tight_layout()
plt.show()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\1686780548.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [105]:
# ── Compare N individual windows with custom range (user-specified) ──────────
N_WINDOWS = 1     # ← change this to select how many paths to visualize
START_IDX = 50       # ← start index (0 = beginning)
END_IDX   = 150   # ← end index (None = full length, or e.g. 500, 1000, etc.)

n_ref = min(N_WINDOWS, len(ref_paths))
n_gen = min(N_WINDOWS, len(gen_paths))

ref_idx = rng.choice(len(ref_paths), size=n_ref, replace=False)
gen_idx = rng.choice(len(gen_paths), size=n_gen, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Reference paths ─────────────────────────────────────────────────────────
ax = axes[0]
for i in ref_idx:
    p = ref_paths[i][START_IDX:END_IDX]
    ax.plot(np.arange(START_IDX, START_IDX + len(p)), p, 
            alpha=0.4, linewidth=0.9, color="tab:blue")
ax.set_title(f"Reference (N={n_ref}, range [{START_IDX}:{END_IDX}])", 
             fontsize=12, fontweight="bold")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close")
ax.grid(True, linewidth=0.3)

# ── Generated paths ─────────────────────────────────────────────────────────
ax = axes[1]
for i in gen_idx:
    p = gen_paths[i][START_IDX:END_IDX]
    ax.plot(np.arange(START_IDX, START_IDX + len(p)), p, 
            alpha=0.4, linewidth=0.9, color="tab:orange")
ax.set_title(f"Generated (N={n_gen}, range [{START_IDX}:{END_IDX}])", 
             fontsize=12, fontweight="bold")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close")
ax.grid(True, linewidth=0.3)

plt.suptitle(f"Window comparison (N={N_WINDOWS}, range [{START_IDX}:{END_IDX}])", 
             y=1.00, fontsize=13)
plt.tight_layout()
plt.show()

print(f"Plotted {n_ref} reference and {n_gen} generated paths")
print(f"Range: [{START_IDX}:{END_IDX}]")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\3126294916.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Plotted 1 reference and 1 generated paths
Range: [50:150]


## 3. Level distribution: histogram, QQ plot, ECDF

In [106]:
gen_checkpoint_name = os.path.basename(GEN_DIRECTORY.rstrip('/'))
ref_checkpoint_name = os.path.basename(REF_DIRECTORY.rstrip('/'))

In [107]:
ref_flat = np.concatenate(ref_paths)
gen_flat = np.concatenate(gen_paths)

ks_stat_lev, ks_p_lev = scipy_stats.ks_2samp(ref_flat, gen_flat)
print(f"Levels KS statistic = {ks_stat_lev:.4f}  |  p-value = {ks_p_lev:.4e}")

combined = np.concatenate([ref_flat, gen_flat])
lo, hi   = np.quantile(combined, [0.002, 0.998])
bins     = np.linspace(lo, hi, 80)
xs       = np.linspace(lo, hi, 400)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Histogram / density
ax = axes[0]
ax.hist(ref_flat, bins=bins, density=True, alpha=0.45, label="Reference", color="tab:blue")
ax.hist(gen_flat, bins=bins, density=True, alpha=0.45, label="Generated", color="tab:orange")
ax.set_title("Level marginal distribution")
ax.set_xlabel("log_adj_close")
ax.set_ylabel("density")
ax.legend(fontsize=8)

# (b) Empirical QQ plot (ref quantiles vs gen quantiles)
ax = axes[1]
n_q  = min(len(ref_flat), len(gen_flat), 5_000)
probs = np.linspace(0.01, 0.99, n_q)
q_ref = np.quantile(ref_flat, probs)
q_gen = np.quantile(gen_flat, probs)
ax.scatter(q_ref, q_gen, s=3, alpha=0.4, color="steelblue")
lims = [min(q_ref.min(), q_gen.min()), max(q_ref.max(), q_gen.max())]
ax.plot(lims, lims, "r--", linewidth=1.2, label="y = x (perfect)")
ax.set_title("QQ plot — levels\n(ref quantiles vs gen quantiles)")
ax.set_xlabel("Reference quantiles")
ax.set_ylabel("Generated quantiles")
ax.legend(fontsize=8)

# (c) ECDF
ax = axes[2]
for vals, label, color in [
    (ref_flat, "Reference", "tab:blue"),
    (gen_flat, "Generated", "tab:orange")
]:
    s = np.sort(vals)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), label=label, linewidth=1.2)
ax.set_xlim(lo, hi)
ax.set_title(f"ECDF — levels  (KS={ks_stat_lev:.3f}, p={ks_p_lev:.2e})")
ax.set_xlabel("log_adj_close")
ax.set_ylabel("cumulative probability")
ax.legend(fontsize=8)

plt.suptitle("log_adj_close — level distributions", y=1.02)
plt.tight_layout()
plt.savefig(f"../images/comparison/level_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
plt.show()

Levels KS statistic = 0.0464  |  p-value = 0.0000e+00


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\31005602.py:52: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\31005602.py:53: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(f"../images/comparison/level_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\31005602.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Increment distribution: histogram, QQ plot, ECDF

In [108]:
ref_incs = np.concatenate([np.diff(p) for p in ref_paths if len(p) > 1])
gen_incs = np.concatenate([np.diff(p) for p in gen_paths if len(p) > 1])

ks_stat_inc, ks_p_inc = scipy_stats.ks_2samp(ref_incs, gen_incs)
print(f"Increments KS statistic = {ks_stat_inc:.4f}  |  p-value = {ks_p_inc:.4e}")

combined_inc = np.concatenate([ref_incs, gen_incs])
lo_i, hi_i  = np.quantile(combined_inc, [0.002, 0.998])
bins_i      = np.linspace(lo_i, hi_i, 80)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Histogram / density
ax = axes[0]
ax.hist(ref_incs, bins=bins_i, density=True, alpha=0.45, label="Reference", color="tab:blue")
ax.hist(gen_incs, bins=bins_i, density=True, alpha=0.45, label="Generated", color="tab:orange")
ax.set_title("Increment marginal distribution")
ax.set_xlabel("ΔX = log_adj_close[t] − log_adj_close[t−1]")
ax.set_ylabel("density")
ax.legend(fontsize=8)

# (b) Empirical QQ plot
ax = axes[1]
n_q  = min(len(ref_incs), len(gen_incs), 5_000)
probs = np.linspace(0.01, 0.99, n_q)
q_ref_i = np.quantile(ref_incs, probs)
q_gen_i = np.quantile(gen_incs, probs)
ax.scatter(q_ref_i, q_gen_i, s=3, alpha=0.4, color="steelblue")
lims_i = [min(q_ref_i.min(), q_gen_i.min()), max(q_ref_i.max(), q_gen_i.max())]
ax.plot(lims_i, lims_i, "r--", linewidth=1.2, label="y = x (perfect)")
ax.set_title("QQ plot — increments\n(ref quantiles vs gen quantiles)")
ax.set_xlabel("Reference quantiles")
ax.set_ylabel("Generated quantiles")
ax.legend(fontsize=8)

# (c) ECDF
ax = axes[2]
for vals, label, color in [
    (ref_incs, "Reference", "tab:blue"),
    (gen_incs, "Generated", "tab:orange")
]:
    s = np.sort(vals)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), label=label, linewidth=1.2)
ax.set_xlim(lo_i, hi_i)
ax.set_title(f"ECDF — increments  (KS={ks_stat_inc:.3f}, p={ks_p_inc:.2e})")
ax.set_xlabel("ΔX")
ax.set_ylabel("cumulative probability")
ax.legend(fontsize=8)

plt.suptitle("Increments ΔX_t — distributions", y=1.02)
plt.tight_layout()
plt.savefig(f"../images/comparison/increments_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
plt.show()

Increments KS statistic = 0.0423  |  p-value = 0.0000e+00


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\895376522.py:51: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\895376522.py:52: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(f"../images/comparison/increments_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\895376522.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Sample paths

Visual check: do the shapes, scale, and fan-out look alike?

Reference paths may be longer; if so, a random window of the generated-path length is
extracted (using `SEED`) for a fair apples-to-apples visual.

In [109]:
N_SHOW   = 20
GEN_LEN  = int(np.median(gen_lens))   # typical generated path length

# Sample random paths from each dataset
ref_idx = rng.choice(len(ref_paths), size=min(N_SHOW, len(ref_paths)), replace=False)
gen_idx = rng.choice(len(gen_paths), size=min(N_SHOW, len(gen_paths)), replace=False)

def extract_window(path: np.ndarray, window_len: int, rng: np.random.Generator) -> np.ndarray:
    """Extract a random window of `window_len` from `path`."""
    if len(path) <= window_len:
        return path
    start = rng.integers(0, len(path) - window_len + 1)
    return path[start : start + window_len]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

# Reference
ax = axes[0]
for i in ref_idx:
    p = extract_window(ref_paths[i], GEN_LEN, rng)
    p = p - p[0]   # anchor at 0
    ax.plot(np.arange(len(p)), p, alpha=0.35, linewidth=0.8, color="tab:blue")
ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_title(f"Reference paths (n={len(ref_idx)}, window={GEN_LEN} steps, anchored at 0)")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close − log_adj_close[0]")

# Generated
ax = axes[1]
for i in gen_idx:
    p = gen_paths[i]
    p = p - p[0]   # anchor at 0
    ax.plot(np.arange(len(p)), p, alpha=0.35, linewidth=0.8, color="tab:orange")
ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_title(f"Generated paths (n={len(gen_idx)}, len={GEN_LEN} steps, anchored at 0)")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close − log_adj_close[0]")

plt.suptitle("Sample paths comparison (all anchored at 0)", y=1.02)
plt.tight_layout()
plt.savefig
plt.savefig(f"../images/comparison/sample_paths_comparison_{gen_checkpoint_name}.png", dpi=300)
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\2721214629.py:15: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_17932\2721214629.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Autocorrelation: levels vs increments

**Rationale**:
- *Levels*: log-price paths should show high persistence (ACF ≈ 1, slow decay).
- *Increments*: log-returns should be nearly i.i.d. (ACF ≈ 0 at all lags ≥ 1).

We average ACFs across multiple paths for a stable estimate.  
For reference paths longer than `GEN_LEN` we extract the first `GEN_LEN` steps.

In [110]:
# N_ACF_PATHS = min(50, len(ref_paths), len(gen_paths))
# LAGS        = min(30, GEN_LEN // 2)

# def mean_acf_across_paths(
#     paths: list[np.ndarray],
#     n_paths: int,
#     lags: int,
#     use_diff: bool = False,
#     max_len: int | None = None,
#     rng: np.random.Generator | None = None,
# ) -> np.ndarray:
#     """Average ACF (lags 1..lags) across up to `n_paths` paths."""
#     indices = (
#         rng.choice(len(paths), size=n_paths, replace=False)
#         if rng is not None
#         else np.arange(min(n_paths, len(paths)))
#     )
#     acfs = []
#     for i in indices:
#         x = paths[i]
#         if max_len is not None and len(x) > max_len:
#             x = x[:max_len]   # align to generated path length
#         if use_diff:
#             x = np.diff(x)
#         if len(x) <= lags:
#             continue
#         acfs.append(sm_acf(x, nlags=lags, fft=True)[1:])   # skip lag-0 (=1)
#     return np.mean(acfs, axis=0) if acfs else np.zeros(lags)

# lag_axis = np.arange(1, LAGS + 1)

# ref_acf_lev = mean_acf_across_paths(ref_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=False, max_len=GEN_LEN, rng=rng)
# gen_acf_lev = mean_acf_across_paths(gen_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=False, max_len=None, rng=rng)
# ref_acf_inc = mean_acf_across_paths(ref_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=True,  max_len=GEN_LEN, rng=rng)
# gen_acf_inc = mean_acf_across_paths(gen_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=True,  max_len=None, rng=rng)

# fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ax = axes[0]
# ax.plot(lag_axis, ref_acf_lev, label="Reference", color="tab:blue",   linewidth=1.4)
# ax.plot(lag_axis, gen_acf_lev, label="Generated", color="tab:orange", linewidth=1.4)
# ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
# ax.set_title("ACF of levels  (should be ≈ 1, slow decay)")
# ax.set_xlabel("lag")
# ax.set_ylabel("autocorrelation")
# ax.set_ylim(-0.2, 1.05)
# ax.legend(fontsize=9)

# ax = axes[1]
# ax.plot(lag_axis, ref_acf_inc, label="Reference", color="tab:blue",   linewidth=1.4)
# ax.plot(lag_axis, gen_acf_inc, label="Generated", color="tab:orange", linewidth=1.4)
# ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
# ax.set_title("ACF of increments  (should be ≈ 0 at all lags)")
# ax.set_xlabel("lag")
# ax.set_ylabel("autocorrelation")
# ax.set_ylim(-0.3, 1.05)
# ax.legend(fontsize=9)

# plt.suptitle(f"Autocorrelation structure  (avg over {N_ACF_PATHS} paths)", y=1.02)
# plt.tight_layout()
# plt.show()

# print(f"Ref  level  ACF at lag-1: {ref_acf_lev[0]:.3f}  |  "
#       f"Gen  level  ACF at lag-1: {gen_acf_lev[0]:.3f}")
# print(f"Ref  inc    ACF at lag-1: {ref_acf_inc[0]:.3f}  |  "
#       f"Gen  inc    ACF at lag-1: {gen_acf_inc[0]:.3f}")

# 8. Paper's FTS evaluations metrics
These metrics are written to work on **LOG-RETURN**.

In [111]:
sys.path.append(os.path.abspath(".."))

import replication.stylized_facts as sf
import numpy as np

In [112]:
# Reference: full stock series (not windows)
ref_full = {}
for csv_path in sorted(Path(REF_DIRECTORY).glob("*.csv")):
    df = pd.read_csv(csv_path)
    ref_full[csv_path.stem] = df["log_adj_close"].to_numpy(dtype=float)

ref_full_obj    = np.empty(len(ref_full), dtype=object)
ref_full_pooled = np.concatenate(list(ref_full.values()))
for i, r in enumerate(ref_full.values()):
    ref_full_obj[i] = r

# Generated: already 2048-pt independent samples — use as-is
# gen_paths_obj and gen_paths_arr unchanged


In [113]:
# Use sf.distribution() with normalize=True and scale='log'
output_dir = Path("../images/stylized_facts_output_heavy_tails")
output_dir.mkdir(exist_ok=True, parents=True)


In [114]:
print(gen_paths[1][:5], "\n", gen_paths[0][-5:])

[ 0.0068345   0.02234219 -0.01495026 -0.00681712  0.03150266] 
 [ 0.01448682  0.00201755  0.00087603 -0.00508903  0.00203821]


In [115]:
# Prepare data as object arrays (one array per path)
# ref_paths_arr = np.array(ref_paths, dtype=object)
# gen_paths_arr = np.array(gen_paths, dtype=object)

# Using np.concatenate
# ref_paths_arr = np.concatenate(ref_paths)
gen_paths_arr = np.concatenate(gen_paths)

# If ref_paths and gen_paths are lists:
# ref_paths_obj = np.empty(len(ref_paths), dtype=object)
# for i, r in enumerate(ref_paths):
#     ref_paths_obj[i] = r

gen_paths_obj = np.empty(len(gen_paths), dtype=object)
for i, g in enumerate(gen_paths):
    gen_paths_obj[i] = g

# Extract checkpoint names (last part after "/")
gen_checkpoint_name = os.path.basename(GEN_DIRECTORY.rstrip('/'))
ref_checkpoint_name = os.path.basename(REF_DIRECTORY.rstrip('/'))

In [116]:
# # print(ref_paths[:5], "\n", ref_paths.shape)
# print(ref_paths_arr[:5],"\n", ref_paths_arr.shape)
print(gen_paths_obj[:5],"\n", gen_paths_obj.shape)

[array([ 4.98047050e-03, -7.94238700e-03,  4.01039080e-04,  2.67584970e-03,
         5.09344040e-03,  2.44796040e-03,  8.93179500e-03,  1.85107250e-02,
        -7.61501070e-03, -1.60432140e-02,  9.63440300e-03,  2.23004500e-02,
        -5.14742700e-03,  9.11864400e-03,  9.27172400e-03,  7.50531950e-03,
         1.57352020e-03,  1.16501560e-02, -7.35682200e-03, -1.25543050e-02,
        -9.83163100e-03,  5.08554500e-04,  1.45896210e-02, -7.03278320e-03,
        -9.96334400e-03, -8.64645650e-03,  4.72348770e-04, -1.65781470e-02,
         5.72861870e-03, -1.84945300e-02, -9.33635900e-03, -4.67544500e-03,
         1.61066380e-03, -4.37519600e-04, -1.15020570e-02,  6.74017240e-03,
        -1.49799720e-02, -1.62421100e-02, -1.05121160e-02, -5.40273600e-03,
        -4.11676430e-03, -8.20754100e-03,  3.22533140e-02, -3.67636970e-04,
        -1.49867890e-02, -3.44489140e-03, -6.74543000e-03,  7.18391850e-04,
        -9.39056200e-03, -2.60443870e-03,  3.12575000e-02,  1.05818860e-02,
        -3.3

In [117]:
_dist_file = output_dir / f"generated_distribution_{ref_checkpoint_name}.png"
if not _dist_file.exists():
    sf.distribution(
        ref_full_pooled,
        file_name=str(output_dir / f"generated_distribution_{ref_checkpoint_name}"),
        scale="log",
        multiple=False,
        normalize=True,
        granuality=100,
    )

In [118]:
gen_checkpoint_short = f"ep{gen_checkpoint_name.split('ep-')[1].split('_')[0]}_sde-{gen_checkpoint_name.split('sde-')[1].split('_')[0]}"

_dist_file = output_dir / f"generated_distribution_{gen_checkpoint_short}.png"
if not _dist_file.exists():
    sf.distribution(
        gen_paths_arr,
        file_name=str(output_dir / f"generated_distribution_{gen_checkpoint_short}"),
        scale="log",
        multiple=False,
        normalize=True,
        granuality=100,
    )

In [119]:
_dist_file = output_dir / f"reference_volatility_clustering_{ref_checkpoint_name}.png"
if not _dist_file.exists():
    sf.acf(
        ref_full_obj,
        file_name=str(output_dir / f"reference_volatility_clustering_{ref_checkpoint_name}"),
        for_abs=True,
        multiple=True,
        fit=False,
        scale="log",
        max_lag=1000,
    )


In [120]:
_dist_file = output_dir / f"reference_volatility_clustering_{gen_checkpoint_short}.png"
if not _dist_file.exists():
    sf.acf(
        gen_paths_obj,
        file_name=str(output_dir / f"generated_volatility_clustering_{gen_checkpoint_short}"),
        for_abs=True,
        multiple=True,
        fit=False,
        scale="log",
        max_lag=1000,
    )

c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Lenovo\anaconda3\envs\thesis\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


In [121]:
_dist_file = output_dir / f"reference_leverage_effect_{ref_checkpoint_name}.png"
if not _dist_file.exists():
    ref_lev = sf.leverage_effect(
        ref_full_obj,
        file_name=str(output_dir / f"reference_leverage_effect_{ref_checkpoint_name}"),
        multiple=True,
        min_lag=1,
        max_lag=100,
    )

_dist_file = output_dir / f"generated_leverage_effect_{gen_checkpoint_short}.png"
if not _dist_file.exists():
    gen_lev = sf.leverage_effect(
        gen_paths_obj,
        file_name=str(output_dir / f"generated_leverage_effect_{gen_checkpoint_short}"),
        multiple=True,
        min_lag=1,
        max_lag=100,
    )

In [122]:
import powerlaw

def fit_powerlaw(returns, max_sample=100000, seed=55):
    x = np.abs(np.asarray(returns, dtype=float))
    x = x[np.isfinite(x) & (x > 0)]

    if len(x) > max_sample:
        rng_local = np.random.default_rng(seed)
        x = rng_local.choice(x, size=max_sample, replace=False)

    f = powerlaw.Fit(x, discrete=False, verbose=True,
                     parameter_ranges={"alpha": [1.5, 10.0]})

    x_tail = x[x >= f.power_law.xmin]
    alpha_mle = 1 + len(x_tail) / np.sum(np.log(x_tail / f.power_law.xmin))

    return {
        "alpha"    : f.power_law.alpha,
        "alpha_mle": alpha_mle,
        "xmin"     : f.power_law.xmin,
        "ks"       : f.power_law.D,
        "n_total"  : len(x),
        "n_tail"   : len(x_tail),
    }


for paths, label in [(gen_paths, "Generated")]: #[(list(ref_full.values()), "Reference"), (gen_paths, "Generated")]:
    incs = np.concatenate(list(paths))
    r = fit_powerlaw(incs)
    print(f"{label}")
    print(f"  alpha (powerlaw) : {r['alpha']:.4f}")
    print(f"  alpha (MLE)      : {r['alpha_mle']:.4f}")
    print(f"  xmin             : {r['xmin']:.6f}")
    print(f"  KS distance      : {r['ks']:.4f}")
    print(f"  n_total          : {r['n_total']:,}")
    print(f"  n_tail           : {r['n_tail']:,}")
    print()

Calculating best minimal value for power law fit


Fitting xmin: 100%|██████████| 99891/99891 [29:14<00:00, 56.92it/s] 

Generated
  alpha (powerlaw) : 5.0254
  alpha (MLE)      : 5.0254
  xmin             : 0.051021
  KS distance      : 0.0199
  n_total          : 100,000
  n_tail           : 2,096

